### OC22 LMDB Dataset Tutorial

This notebook provides an overview of how to create LMDB datasets to be used with the OCP repo. This tutorial is intended for those who wish to use OCP to train on their own datasets. Those interested in just using OCP data need not worry about these steps as they've been automated as part of the download script: https://github.com/Open-Catalyst-Project/ocp/blob/master/scripts/download_data.py.

In [1]:
import sys
sys.path.append(r"D:\Train_thử\fairchem-tio2-s2ef")

from ocpmodels.preprocessing import AtomsToGraphs
from ocpmodels.datasets import SinglePointLmdbDataset, TrajectoryLmdbDataset, OC22LmdbDataset, LmdbDataset
import ase.io
from ase.build import bulk
from ase.build import fcc100, add_adsorbate, molecule
from ase.constraints import FixAtoms
from ase.calculators.emt import EMT
from ase.optimize import BFGS
import matplotlib.pyplot as plt
import lmdb
import pickle
from tqdm import tqdm
import torch
import os

In [2]:
import sys
print(sys.executable)

c:\Users\admin\anaconda3\envs\tio2-s2ef\python.exe


### Structure to Energy and Forces (S2EF) LMDBs

S2EF LMDBs utilize the TrajectoryLmdb dataset. This dataset expects a directory of LMDB files. In addition to the attributes defined by AtomsToGraph, the following attributes must be added for the S2EF task:

In [2]:
# dataset = TrajectoryLmdbDataset({"src": "s2ef/"})
# dataset = OC22LmdbDataset({"src": r"D:\Train_thử\fairchem-tio2-s2ef\tio2_s2ef\oc22_data\s2ef_total_train_val_test_lmdbs\data\oc22\s2ef-total\val_ood\data.0000.lmdb"})
dataset = LmdbDataset({"src": r"D:\Train_thử\fairchem-tio2-s2ef\tio2_s2ef\oc22_data\tio2_filtered\train"})

len(dataset)


2097

#### Advanced usage

TrajectoryLmdbDataset supports multiple LMDB files because the need to highly parallelize the dataset construction process. With OCP's largest split containing 135M+ frames, the need to parallelize the LMDB generation process for these was necessary. If you find yourself needing to deal with very large datasets we recommend parallelizing this process.

### 1. Interacting with the LMDBs

Below we demonstrate how to interact with an LMDB to extract particular information.

In [3]:
data = dataset[5]
data

Data(y=-805.00991662, pos=[92, 3], cell=[1, 3, 3], atomic_numbers=[92], natoms=92, force=[92, 3], fixed=[92], tags=[92], nads=1, sid=23315, fid=351, id='0_5', oc22=1)

#### 1.1 Check specific sid in dataset

In [4]:
target_sid = 23315
matches = []
for i in range(len(dataset)):
    try:
        if dataset[i].sid == target_sid:
            matches.append(i)
    except Exception:
        continue

print(f"Found {len(matches)} matching indices for sid={target_sid}: {matches}")


Found 473 matching indices for sid=23315: [4, 5, 12, 16, 17, 25, 27, 41, 42, 45, 56, 64, 72, 73, 74, 78, 80, 81, 91, 93, 98, 105, 108, 109, 110, 115, 124, 132, 139, 141, 142, 146, 150, 153, 155, 157, 164, 166, 168, 169, 189, 200, 202, 206, 211, 215, 218, 221, 227, 231, 233, 238, 239, 242, 244, 246, 247, 250, 252, 254, 260, 264, 273, 274, 279, 284, 288, 289, 304, 305, 308, 310, 321, 322, 327, 331, 338, 344, 350, 358, 363, 371, 372, 374, 384, 393, 397, 400, 403, 409, 410, 411, 412, 419, 420, 434, 435, 439, 440, 442, 444, 446, 448, 452, 457, 459, 460, 461, 464, 468, 469, 473, 486, 489, 495, 498, 499, 500, 506, 510, 524, 528, 530, 540, 542, 552, 559, 560, 563, 573, 577, 586, 587, 592, 598, 600, 604, 613, 614, 618, 619, 620, 621, 626, 634, 637, 640, 645, 647, 653, 664, 669, 672, 675, 676, 677, 689, 699, 701, 702, 705, 707, 708, 711, 717, 724, 742, 744, 746, 750, 752, 762, 769, 772, 773, 774, 781, 783, 789, 791, 793, 800, 802, 805, 818, 821, 838, 839, 841, 843, 846, 856, 860, 862, 863, 865, 

In [5]:
data = dataset[5]
data

Data(y=-805.00991662, pos=[92, 3], cell=[1, 3, 3], atomic_numbers=[92], natoms=92, force=[92, 3], fixed=[92], tags=[92], nads=1, sid=23315, fid=351, id='0_5', oc22=1)

In [6]:
data.atomic_numbers[90]
data.force[90]

tensor([ 0.0246, -0.0035, -0.0158])

In [7]:
from ase.data import chemical_symbols

# Assuming 'data' is your dataset[0]
atom_list = [chemical_symbols[int(z)] for z in data.atomic_numbers]

print(f"Elements in this system: {atom_list}")

# To see which atoms are the 'adsorbate' (molecule) vs 'surface'
for i, symbol in enumerate(atom_list):
    tag = data.tags[i].item()
    role = "Subsurface" if tag == 0 else "Surface" if tag == 1 else "Adsorbate"
    print(f"Atom {i}: {symbol} ({role}) at {data.pos[i].tolist()}")

Elements in this system: ['Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'Ti', 'C', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Atom 0: Ti (Subsurface) at [2.7031970024108887, 6.856969356536865, 12.525662422180176]
Atom 1: Ti (Subsurface) at [8.774721145629883, 2.128875494003296, 12.758586883544922]
Atom 2: Ti (Subsurface) at [5.541818618774414, 0.19577501714229584, 13.017812728881836]
Atom 3: Ti (Subsurface) at [-0.41897740960121155, 4.662298202514648, 12.794341087341309]
Atom 4: Ti (Subsurface) at [2.922197103500366, 0.38487139344215393, 12.438848495483398]
Atom 5: Ti (Subsurface) at [6.83956575393676

#### 1.2 Drawing 3D of the data

In [10]:
!pip install plotly

You should consider upgrading via the 'C:\Users\admin\AppData\Local\Programs\Python\Python39\python.exe -m pip install --upgrade pip' command.


In [11]:
import plotly.graph_objects as go
from ase.data import chemical_symbols

# Data from current sample
pos = data.pos.numpy()
z = data.atomic_numbers.numpy().astype(int)
tags = data.tags.numpy().astype(int)

# Roles by tag
roles = {0: "Subsurface", 1: "Surface", 2: "Adsorbate"}
colors = {"Subsurface": "blue", "Surface": "green", "Adsorbate": "red"}

fig = go.Figure()

for tag_val, role in roles.items():
    mask = tags == tag_val
    if mask.sum() == 0:
        continue
    pts = pos[mask]
    fig.add_trace(
        go.Scatter3d(
            x=pts[:, 0],
            y=pts[:, 1],
            z=pts[:, 2],
            mode="markers+text",
            marker=dict(size=4, color=colors[role]),
            name=f"{role} ({mask.sum()})",
            text=[chemical_symbols[n] for n in z[mask]],
            textposition="top center",
            hovertemplate="atom=%{text}<br>x=%{x:.2f}<br>y=%{y:.2f}<br>z=%{z:.2f}<extra></extra>",
        )
    )

fig.update_layout(
    title=f"3D Atoms for sid={data.sid}, fid={data.fid}",
    scene=dict(
        xaxis_title="x (Å)",
        yaxis_title="y (Å)",
        zaxis_title="z (Å)",
        aspectmode="data",
    ),
    width=900,
    height=700,
)
fig.show()

#### 2.2 Total Energy

In [8]:
energies = torch.tensor([data.y for data in dataset])
len(energies)

2097

In [9]:
plt.hist(energies, bins = 10)
plt.yscale("log")
plt.xlabel("Energies")
plt.show()

: 

### Visualization

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [ ]:
from ase import Atoms
from ase.visualize import view
import torch

# 1. Extract the information from your Data object
# We reshape the cell from [1, 3, 3] to [3, 3]
positions = data.pos.numpy()
numbers = data.atomic_numbers.numpy().astype(int)
cell = data.cell.view(3, 3).numpy()

# 2. Create the ASE Atoms object
structure = Atoms(
    numbers=numbers,
    positions=positions,
    cell=cell,
    pbc=True # Periodic Boundary Conditions are standard for OCP
)

# 3. Open the visualizer
view(structure)

# Optional: Save it as a 3D file for VESTA or Ovito
structure.write("system_2090899_frame_206.xyz")

### 3D Visualization

In [ ]:
import ase.io
from ase import Atoms
import torch

# 1. Access the specific data object (e.g., sid 2090899, fid 206)
data = dataset[0] 

# 2. Convert to ASE Atoms object
# Reshape the cell to [3, 3] as required by ASE
atoms = Atoms(
    numbers=data.atomic_numbers.numpy().astype(int),
    positions=data.pos.numpy(),
    cell=data.cell.view(3, 3).numpy(),
    pbc=True
)

# 3. Attach metadata (This makes it "Extended")
# We store energy and forces so other models or Ovito can read them
atoms.info["energy"] = data.y.item() if isinstance(data.y, torch.Tensor) else data.y
atoms.arrays["forces"] = data.force.numpy()
atoms.info["sid"] = str(data.sid)
atoms.info["fid"] = str(data.fid)

# 4. Save as ExtXYZ
ase.io.write("system_data.extxyz", atoms, format="extxyz")

In [ ]:
from ase.visualize import view
import ase.io

# 1. Đọc dữ liệu từ file ExtXYZ bạn đã tạo
atoms = ase.io.read("system_data.extxyz")

# 2. Hiển thị bằng viewer 'x3d' (nó sẽ hiện ra một cửa sổ tương tác 3D)
view(atoms, viewer='x3d')